<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent; border-radius: 14px; padding: 18px 22px; margin: 12px 0;
  font-size: 36px; font-weight: 600; color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25); background-clip: padding-box; position: relative;
">
  <div style="position: absolute; inset: 0; padding: 4px; border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: linear-gradient(#fff 0 0) content-box, linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor; mask-composite: exclude; pointer-events: none;"></div>
  <b>01. Bayesian Time Series Forecasting</b>
</div>

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">📑 Table of Contents</span>

- **1. [Advanced STS: Multiple Seasonalities](#-part-i-advanced-sts)**
- **2. [Setup & Imports](#-part-ii-setup--imports)**
- **3. [BSTS with Regression Components](#-part-iii-bsts-regression)**
    - 3.1 [Adding External Regressors](#external-regressors)
    - 3.2 [Spike-and-Slab Variable Selection](#spike-and-slab)
- **4. [MCMC-Based Inference for Time Series](#-part-iv-mcmc-inference)**
- **5. [Forecast Evaluation & Calibration](#-part-v-forecast-evaluation)**
    - 5.1 [CRPS and Calibration Metrics](#crps)
    - 5.2 [Coverage Probability](#coverage)
- **6. [Key Takeaways](#-part-vi-key-takeaways)**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">1. Advanced STS: Multiple Seasonalities</span>

Real-world time series often have **multiple seasonal patterns**:

| **Pattern** | **Period** | **Example** |
|-----------|----------|-------------|
| Daily | 24 hours | Electricity demand |
| Weekly | 7 days | Website traffic |
| Monthly | ~30 days | Sales cycles |
| Yearly | 365 days | Weather, holidays |

TFP's `sts.Sum` lets you combine multiple seasonal components with different periods.

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">2. Setup & Imports</span>

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt

tfd = tfp.distributions
sts = tfp.sts

tf.random.set_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print(f"TensorFlow: {tf.__version__}")
print(f"TFP: {tfp.__version__}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">3. BSTS with Regression Components</span>

In [ ]:
# Generate complex time series: trend + two seasonalities + regression + noise
np.random.seed(42)
T = 365
t = np.arange(T).astype(np.float32)

trend = 0.02 * t + 5.0
season_weekly = 2.0 * np.sin(2 * np.pi * t / 7)
season_yearly = 4.0 * np.sin(2 * np.pi * t / 365)

# Exogenous regressor (e.g., temperature)
temperature = (20 + 10 * np.sin(2 * np.pi * t / 365 - np.pi/4) + 
               np.random.normal(0, 2, T)).astype(np.float32)
temp_effect = 0.3 * temperature

noise = np.random.normal(0, 1.0, T).astype(np.float32)
y = (trend + season_weekly + season_yearly + temp_effect + noise).astype(np.float32)

# Train/test split
train_size = 300
y_train = y[:train_size]
y_test = y[train_size:]
temp_train = temperature[:train_size]
temp_test = temperature[train_size:]

observed = tf.constant(y_train[..., tf.newaxis])

# Build multi-component STS model
model = sts.Sum([
    sts.LocalLinearTrend(observed_time_series=observed, name='trend'),
    sts.Seasonal(num_seasons=7, observed_time_series=observed, name='weekly'),
    sts.Seasonal(num_seasons=52, num_steps_per_season=7, 
                 observed_time_series=observed, name='yearly'),
    sts.LinearRegression(
        design_matrix=temp_train[..., tf.newaxis, tf.newaxis],
        name='temperature'
    ),
], observed_time_series=observed)

print("Model components:")
for c in model.components:
    print(f"  - {c.name}")
print(f"Total parameters: {sum(p.shape.num_elements() for p in model.parameters)}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">4. MCMC-Based Inference for Time Series</span>

In [ ]:
# Fit using HMC for full posterior
@tf.function
def fit_with_hmc():
    return tfp.sts.fit_with_hmc(
        model=model,
        observed_time_series=observed,
        num_results=200,
        num_warmup_steps=100,
        num_variational_steps=100
    )

hmc_samples, kernel_results = fit_with_hmc()

print("HMC posterior samples obtained!")
for param, samples in zip(model.parameters, hmc_samples):
    mean_val = tf.reduce_mean(samples).numpy()
    std_val = tf.math.reduce_std(samples).numpy()
    print(f"  {param.name}: {mean_val:.4f} ± {std_val:.4f}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">5. Forecast Evaluation & Calibration</span>

In [ ]:
# Generate probabilistic forecast
forecast_dist = tfp.sts.forecast(
    model=model,
    observed_time_series=observed,
    parameter_samples=hmc_samples,
    num_steps_forecast=len(y_test)
)

forecast_mean = forecast_dist.mean().numpy().mean(axis=0)[:, 0]
forecast_std = forecast_dist.stddev().numpy().mean(axis=0)[:, 0]

# Coverage probability
lower_95 = forecast_mean - 1.96 * forecast_std
upper_95 = forecast_mean + 1.96 * forecast_std
coverage = np.mean((y_test >= lower_95) & (y_test <= upper_95))

# MAE and RMSE
mae = np.mean(np.abs(y_test - forecast_mean))
rmse = np.sqrt(np.mean((y_test - forecast_mean)**2))

print(f"Forecast Metrics:")
print(f"  MAE:  {mae:.3f}")
print(f"  RMSE: {rmse:.3f}")
print(f"  95% Coverage: {coverage:.1%} (ideal: 95%)")

# Visualize
fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(t[:train_size], y_train, color='#3b82f6', linewidth=1, alpha=0.7, label='Training')
ax.plot(t[train_size:], y_test, color='#1e293b', linewidth=2, label='True (held-out)')
ax.plot(t[train_size:], forecast_mean, color='#ef4444', linewidth=2, label='Forecast mean')
ax.fill_between(t[train_size:], lower_95, upper_95, alpha=0.2, color='#ef4444', label='95% CI')
ax.axvline(x=train_size, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Time', fontsize=14)
ax.set_ylabel('Value', fontsize=14)
ax.set_title(f'Bayesian Time Series Forecast\nCoverage: {coverage:.1%}, RMSE: {rmse:.2f}',
             fontsize=16, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">6. Key Takeaways</span>

| **Concept** | **Key Point** |
|------------|---------------|
| Multiple seasonalities | Use multiple `sts.Seasonal` with different periods |
| Regression components | Add exogenous variables via `sts.LinearRegression` |
| HMC inference | Full posterior via `tfp.sts.fit_with_hmc` |
| Coverage probability | Calibration metric — should match nominal level |
| CRPS | Proper scoring rule for probabilistic forecasts |

### Next Steps
- **Notebook 02**: Combine deep learning (RNNs/LSTMs) with probabilistic layers for sequence modeling